# Bronze → Silver

Conceitos praticados:
- **Padronização**: todas as colunas em português e com a tipagem correta
- **Deduplicação** por chave de negócio
- **Conversão segura de tipos**: textos fora de contexto (Column Shift) viram `NULL` sem interromper o pipeline
- **Regras de domínio**: escala de notas, valores negativos, status válidos

As tabelas Bronze são apenas lidas; nenhuma alteração é feita nelas.

In [0]:
catalog = "cinedata_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

DataFrame[]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Datas que não casam com o padrão testado viram NULL em vez de gerar erro de versão do parser
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

def to_double(col_name: str):
    """Conversão segura: só converte quando o texto é um número, caso contrário retorna NULL."""
    valor = F.trim(F.col(col_name))
    return F.when(valor.rlike(r"^-?\d+(\.\d+)?$"), valor.cast("double"))

def to_int(col_name: str):
    """Conversão segura para inteiro: valores com texto ou casas decimais viram NULL."""
    valor = F.trim(F.col(col_name))
    return F.when(valor.rlike(r"^-?\d+$"), valor.cast("int"))

## 1. tb_info_filmes
- **Deduplicação**: uma linha por filme, mantendo a versão mais recente pela `ingestion_datetime`
- **Status**: antes da tradução, a coluna é normalizada (caixa baixa, hífens e ruídos viram espaço). Valores corrompidos ou não mapeáveis viram `"Não Informado"`
- **Data multi-formato**: a origem mistura `yyyy-MM-dd`, `dd/MM/yyyy` e `MM-dd-yyyy`. Testamos os três padrões e só fica `NULL` o que não converte em nenhum deles
- Coluna derivada `ano_lancamento`

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

janela_mais_recente = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_silver_info = (
    df_bronze_info
    .withColumn("rn", F.row_number().over(janela_mais_recente))
    .filter(F.col("rn") == 1)
    .withColumn("status", F.trim(F.regexp_replace(F.lower(F.col("status")), r"[^a-z]+", " ")))
    .withColumn("status",
        F.when(F.col("status") == "released", "Lançado")
        .when(F.col("status") == "post production", "Pós-Produção")
        .when(F.col("status") == "in production", "Em Produção")
        .when(F.col("status") == "planned", "Planejado")
        .when(F.col("status") == "rumored", "Rumores")
        .when(F.col("status") == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )
    .withColumn("release_date", F.coalesce(
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("dd/MM/yyyy"))),
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("MM-dd-yyyy"))),
    ))
    .withColumn("runtime", to_int("runtime"))
    .select("id", "title", "original_title", "release_date", "runtime",
            "original_language", "status", "overview", "tagline")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

display(df_silver_info.limit(10))

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,2022-07-07,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,0,es,Lançado,null,null,2021
1000079,Pourquoi tu souris ?,Pourquoi tu souris ?,2024-06-26,0,fr,Em Produção,null,null,2024
1000099,Chowdhury Rajbari,Chowdhury Rajbari,2017-12-08,131,bn,Lançado,"Giaa and Arnab are a married couple, who visit Arnab's ancestral palace 'Chowdhury Rajbari' for a romantic getaway. Once there, Giaa experiences strange occurrences and memories of a past life. In her search for truth, Giaa comes across Dr. Oindrila, who assists her in her search. But trouble starts when they come across more questions, instead of answers.",null,2017
1000110,Les Enfants rouges,Les Enfants rouges,2023-10-25,0,fr,Lançado,null,null,2023
1000116,Don't Wait for Me,אל תחכי לי,2021-10-21,88,he,Lançado,Matan arrives in a crime neighborhood after his father goes bankrupt and loses their home.,null,2021
1000156,The Kids of Santa Fe: The Largest Unknown Mass Shooting,The Kids of Santa Fe: The Largest Unknown Mass Shooting,2021-09-10,120,en,Lançado,On 2018 a student at Santa Fe High School executed 10 people and wounded 13 others. The shooting took half an hour before the killer was arrested. The movie is about the students that were directly affected and what they are doing today.,null,2021
1000517,The Temple Woods Gang,Le gang des bois du temple,2022-10-16,116,fr,Lançado,"A retired military man lives in the Temple Woods housing project. Just as he’s burying his mother, his neighbour Bébé, who belongs to a gang of robbers from the area, is preparing to rob the convoy of a wealthy Arab prince.",null,2022
1000643,Mise au vert,Mise au vert,2023-09-20,0,fr,Lançado,"To strengthen family ties, to free his teenagers from their digital addiction, to reconnect with his roots in the region of his childhood; this is why Régis decides to take his wife and children on a surprise vacation in the Vercors. An old ruined house lost in the forest. No water, no electricity, even less network or wifi. Nothing is going to happen as planned",null,2023
1000780,"You Sing Loud, I Sing Louder","You Sing Loud, I Sing Louder",2023-03-11,102,en,Lançado,"In an effort to reunite with his daughter after a period of absence, a father takes her on a road trip to Santa Fe, New Mexico. Along the way they encounter some interesting characters and obstacles while sorting out their relationship.",null,2023


## 2. tb_financeiro_filmes
- Textos que representam ausência (`N/A`, `Unknown`, `Não Informado`) viram `NULL` antes da conversão
- Higienização de símbolos de moeda (`$`, `USD`) e pontuações de milhar
- Conversão para `DECIMAL(18,2)`; valores zerados ou negativos são tratados como ausentes
- Valores em BRL calculados com a cotação extraída da API (a mais recente gravada em `bronze.tb_cotacao_dolar`)
- **Lucro** só é calculado quando orçamento e receita existem; caso contrário fica `NULL`. A **margem** só é calculada com receita positiva, evitando divisão por zero

In [0]:
cotacao_dolar = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .first()["cotacaoCompra"]
)
print(f"Cotação utilizada: R$ {cotacao_dolar}")

textos_ausentes = ["N/A", "UNKNOWN", "NÃO INFORMADO"]

def limpar_valor_monetario(col_name: str):
    valor = F.upper(F.trim(F.col(col_name)))
    valor = F.when(valor.isin(textos_ausentes), None).otherwise(valor)
    valor = F.regexp_replace(valor, r"USD|\$|,|\s", "")
    return F.when(valor.rlike(r"^-?\d{1,16}(\.\d+)?$"), valor.cast("decimal(18,2)"))

Cotação utilizada: R$ 4.9886


In [0]:
df_bronze_financeiro = spark.table(f"{bronze_schema}.tb_movies_financials")

df_silver_financeiro = (
    df_bronze_financeiro
    .withColumn("budget", limpar_valor_monetario("budget"))
    .withColumn("revenue", limpar_valor_monetario("revenue"))
    .withColumn("budget", F.when(F.col("budget") > 0, F.col("budget")))
    .withColumn("revenue", F.when(F.col("revenue") > 0, F.col("revenue")))
    .select("id", "budget", "revenue")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.lit(cotacao_dolar).cast("decimal(10,4)")).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.lit(cotacao_dolar).cast("decimal(10,4)")).cast("decimal(18,2)"))
    .withColumn("lucro_usd",
        F.when(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull(),
               F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl",
        F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
               F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    .withColumn("margem_lucro_percentual",
        F.when(F.col("lucro_usd").isNotNull() & (F.col("receita_usd") > 0),
               F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast("double"))
)

df_silver_financeiro.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

display(df_silver_financeiro.limit(10))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
293660,58000000.00,null,289338800.00,null,null,null,null
299536,300000000.00,2052415039.00,1496580000.00,10238677663.56,1752415039.00,8742097663.56,85.38
299534,356000000.00,2800000000.00,1775941600.00,13968080000.00,2444000000.00,12192138400.00,87.29
475557,55000000.00,1074458282.00,274373000.00,5360042585.59,1019458282.00,5085669585.59,94.88
271110,250000000.00,null,1247150000.00,null,null,null,null
284054,200000000.00,1349926083.00,997720000.00,6734241257.65,1149926083.00,5736521257.65,85.18
284052,180000000.00,676343174.00,897948000.00,3374005557.82,496343174.00,2476057557.82,73.39
315635,175000000.00,880166924.00,873005000.00,4390800717.07,705166924.00,3517795717.07,80.12
283995,200000000.00,863756051.00,997720000.00,4308933436.02,663756051.00,3311213436.02,76.85
297761,175000000.00,746846894.00,873005000.00,3725720415.41,571846894.00,2852715415.41,76.57


## 3. tb_metricas_engajamento
- **Popularidade**: a origem mistura vírgula e ponto como separador decimal (`154,34` e `72.735`). A vírgula é trocada por ponto antes da conversão, para que o valor não vire `NULL`
- **Conversão segura**: textos deslocados (Column Shift) nas colunas de notas e votos viram `NULL`
- **Regras de negócio**: notas fora do intervalo 0 a 10 (incluindo erros de escala) e contagens ou popularidade negativas viram `NULL`

In [0]:
df_bronze_metricas = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_silver_metricas = (
    df_bronze_metricas
    .withColumn("popularity", F.regexp_replace(F.trim(F.col("popularity")), ",", "."))
    # Linha com Column Shift (texto nas colunas do IMDb): um inteiro na popularidade é um ano fora de contexto
    .withColumn("popularity",
        F.when((F.col("averageRating").rlike("[A-Za-z]") | F.col("numVotes").rlike("[A-Za-z]"))
               & F.col("popularity").rlike(r"^\d+$"), None)
        .otherwise(F.col("popularity")))
    .withColumn("popularidade", to_double("popularity"))
    .withColumn("nota_media_tmdb", to_double("vote_average"))
    .withColumn("qtd_votos_tmdb", to_int("vote_count"))
    .withColumn("nota_media_imdb", to_double("averageRating"))
    .withColumn("qtd_votos_imdb", to_int("numVotes"))
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
    .select(F.col("id").alias("id_filme"), "popularidade", "nota_media_tmdb",
            "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
)

df_silver_metricas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(df_silver_metricas.limit(10))

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.735,7.606,28894,8.0,1270339
299536,154.34,8.255,27713,8.4,1406782
299534,91.756,8.263,23857,8.4,1484150
475557,54.522,8.168,23425,8.3,1723035
271110,70.741,7.4,21541,7.8,947222
284054,43.665,7.39,null,7.3,924922
284052,70.535,7.427,20935,7.5,895880
315635,65.88,7.345,20507,7.4,835116
283995,67.553,7.624,20353,7.6,844767
297761,35.356,5.909,20097,5.9,null


## 4. tb_avaliacoes_usuarios
- Remoção de registros integralmente duplicados (mesmo filme, usuário, nota e comentário)
- `nota_usuario` fora da escala de 0 a 10 é descartada e vira `NULL`
- Comentários vazios ou só com espaços recebem o texto padronizado `"Sem comentário"`

In [0]:
df_bronze_avaliacoes = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_silver_avaliacoes = (
    df_bronze_avaliacoes
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    .withColumn("nota", to_double("nota"))
    .withColumn("nota", F.when(F.col("nota").between(0, 10), F.col("nota")))
    .withColumn("comentario",
        F.when(F.trim(F.coalesce(F.col("comentario"), F.lit(""))) == "", "Sem comentário")
        .otherwise(F.col("comentario")))
    .select("id", "nome", "nota", "comentario")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

df_silver_avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(df_silver_avaliacoes.limit(10))

id_filme,nome_usuario,nota_usuario,comentario_usuario
797814,Matheus Cavalcanti,1.1,Péssimo em todos os sentidos.
800797,Sérgio Almeida 335,7.2,Filme interessante com boas atuações.
1086795,Rodrigo Oliveira 273,9.7,Excepcional! História envolvente e atuações impecáveis.
1054508,Tatiana Oliveira 369,7.7,Legal! Uma boa opção para o fim de semana.
776552,Letícia Lopes,8.4,Filme interessante com boas atuações.
750569,Pedro Souza 611,4.4,Sem comentário
464175,Mônica Cavalcanti,7.7,Filme interessante com boas atuações.
823195,Rodrigo Mendes,2.9,Péssimo em todos os sentidos.
1055332,Gustavo Dias 303,null,Sem comentário
679864,Lucas Freitas 371,9.2,Perfeito! Um dos melhores filmes que já vi.


## 5. tb_generos
- O separador ponto e vírgula vira vírgula antes do `split` + `explode`
- Cada linha passa a representar um único gênero por filme
- Resíduos do Column Shift (valores em branco, numéricos e textos descritivos) são removidos validando contra o domínio de gêneros do TMDB

In [0]:
df_bronze_creditos = spark.table(f"{bronze_schema}.tb_credits_and_tags")

generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]

df_silver_generos = (
    df_bronze_creditos
    .withColumn("genres", F.regexp_replace(F.col("genres"), ";", ","))
    .select(F.col("id").alias("id_filme"), F.explode(F.split(F.col("genres"), ",")).alias("nome_genero"))
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
    .filter(F.col("nome_genero").isin(generos_validos))
    .dropDuplicates(["id_filme", "nome_genero"])
)

df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

display(df_silver_generos.limit(10))

id_filme,nome_genero
271110,Adventure
346364,Horror
321612,Family
260513,Action
343668,Comedy
405774,Thriller
405774,Drama
395992,Science Fiction
427641,Action
68735,Fantasy


## 6. tb_pessoas_empresas
Tabela unificada com quatro tipos de entidade:

| Coluna origem | tipo_entidade |
|---|---|
| `cast` | Ator |
| `directors` | Diretor |
| `writers` | Roteirista |
| `production_companies` | Produtora |

- Mesmo tratamento de separadores da tabela de gêneros, seguido de `split` + `explode`
- Remoção de resíduos do Column Shift: valores em branco ou marcadores de ausência (`N/A`, `Nenhum`, `[]`), valores numéricos (sem nenhuma letra) e textos descritivos (que começam com letra minúscula ou têm mais de 8 palavras, como trechos de sinopse e palavras-chave)
- Capitalização padronizada com `initcap` e eliminação de duplicados

In [0]:
colunas_entidades = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

marcadores_ausencia = ["", "N/A", "NENHUM", "[]"]

df_entidades = None
for coluna, tipo in colunas_entidades.items():
    df_tipo = (
        df_bronze_creditos
        .withColumn(coluna, F.regexp_replace(F.col(coluna), ";", ","))
        .select(F.col("id").alias("id_filme"), F.explode(F.split(F.col(coluna), ",")).alias("nome_entidade"))
        .withColumn("tipo_entidade", F.lit(tipo))
    )
    df_entidades = df_tipo if df_entidades is None else df_entidades.unionByName(df_tipo)

df_silver_pessoas_empresas = (
    df_entidades
    .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
    .filter(~F.upper(F.col("nome_entidade")).isin(marcadores_ausencia))
    .filter(F.col("nome_entidade").rlike(r"[A-Za-zÀ-ÿ]"))
    .filter(~F.col("nome_entidade").rlike(r"^[a-z]"))
    .filter(F.size(F.split(F.col("nome_entidade"), r"\s+")) <= 8)
    .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

display(df_silver_pessoas_empresas.limit(10))

id_filme,nome_entidade,tipo_entidade
296096,Vanessa Kirby,Ator
332562,Sam Elliott,Ator
423108,Patrick Wilson,Ator
399404,Lily James,Ator
453405,Benedict Wong,Ator
324849,Doug Benson,Ator
310307,Justin Randell Brooke,Ator
696806,Walker Scobell,Ator
297761,Joel Kinnaman,Ator
284053,Jeff Goldblum,Ator


## 7. tb_cotacao_dolar
- **Série temporal contínua**: calendário do primeiro dia com cotação até a data de execução, com uma linha por dia (mantendo o lote mais recente quando a API é chamada mais de uma vez)
- **Forward Fill**: dias sem cotação (fins de semana e feriados) recebem o valor do último dia útil disponível

In [0]:
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

janela_lote_recente = Window.partitionBy("data_cotacao").orderBy(F.col("ingestion_datetime").desc())

df_cotacao_dia = (
    df_bronze_cotacao
    .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
    .withColumn("rn", F.row_number().over(janela_lote_recente))
    .filter(F.col("rn") == 1)
    .select("data_cotacao", "cotacao_compra")
)

df_calendario = (
    df_cotacao_dia
    .agg(F.min("data_cotacao").alias("data_inicio"))
    .select(F.explode(F.sequence(F.col("data_inicio"), F.current_date())).alias("data_cotacao"))
)

janela_forward_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao = (
    df_calendario
    .join(df_cotacao_dia, on="data_cotacao", how="left")
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_forward_fill))
    .orderBy("data_cotacao")
)

df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

display(df_silver_cotacao)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2026-09-29,5.2198
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692
2026-10-07,4.9929
2026-10-08,5.0113
